In [1]:
from openpyxl import load_workbook
import pandas as pd
import re

In [2]:
# --- Archivos ---
ARCHIVO_ENTRADA = 'Libro1.xlsx'
HOJA_DATOS = 'BASE DE DATOS'
ARCHIVO_SALIDA = 'interventoria_limpio.xlsx'

# --- Columnas del Excel ---
COL_NOMBRE = 'Nombres y Apellidos'
COL_CEDULA = 'No. Cedula de ciudadanía'
COL_CONTRATO = 'No. Contrato de Interventoría'
COL_ANIO = 'Año del Contrato de Interventoría'
COL_FECHA_INICIO = 'Fecha de Inicio del Contrato de Interventoría'
COL_FECHA_FIN = 'Fecha de Finalización del Contrato de Interventoría'
COL_OBJETO = 'Objeto del Contrato  de Interventoría'
COL_CARGO = 'Cargo que desempeña en el Contrato de Interventoría'
COL_PARTICIPACION = 'Participación TOTAL en el proyecto (%)'
COL_ESTADO = 'Contrato Estado'
COL_INTERVENTORIA = 'INTERVENTORÍA'
COL_UNIDAD = 'UNIDAD EJECUTORA'
COL_REGISTRO = 'REGISTRO'

# --- Validación de filas ---
COLUMNAS_VACIAS = [COL_NOMBRE, COL_CEDULA]
MIN_PALABRAS_OBJETO = 15
LARGO_MAX_CARGO = 120

# --- Valores por defecto ---
FECHA_VACIA = '0000-00-00'
VALOR_SIN_CARGO = 'SIN CARGO'
VALOR_SIN_DEFINIR = 'SIN DEFINIR'

# --- Normalización de nombres ---
TABLA_TILDES = str.maketrans({
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
})

# --- Meses ---
MESES = {
    'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04',
    'MAYO': '05', 'JUNIO': '06', 'JULIO': '07', 'AGOSTO': '08',
    'SEPTIEMBRE': '09', 'SETIEMBRE': '09', 'OCTUBRE': '10',
    'NOVIEMBRE': '11', 'DICIEMBRE': '12',
}

MESES_CORTO = {
    'ENE': '01', 'FEB': '02', 'MAR': '03', 'ABR': '04',
    'MAY': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08',
    'SEP': '09', 'OCT': '10', 'NOV': '11', 'DIC': '12',
}





In [3]:
df = pd.read_excel(ARCHIVO_ENTRADA)
df.columns = df.columns.str.strip()

wb = load_workbook(ARCHIVO_ENTRADA)
ws = wb[HOJA_DATOS]

headers = [
    cell.value.strip() if isinstance(cell.value, str) else cell.value
    for cell in ws[1]
]


In [4]:
col_inicio = headers.index(COL_FECHA_INICIO) + 1
col_fin = headers.index(COL_FECHA_FIN) + 1

def tiene_color(celda):
    fill = celda.fill
    if fill is None:
        return False
    if fill.fill_type is None:
        return False
    color = fill.fgColor.rgb
    if color in [None, '00000000', 'FFFFFFFF']:
        return False
    return True

estados = []
for row in range(2, ws.max_row + 1):
    celda_inicio = ws.cell(row=row, column=col_inicio)
    celda_fin = ws.cell(row=row, column=col_fin)
    if tiene_color(celda_inicio) or tiene_color(celda_fin):
        estados.append('FINALIZO')
    else:
        estados.append('ACTIVO')

df[COL_ESTADO] = estados


In [5]:
df = df.drop(columns=[COL_REGISTRO], errors='ignore')
df = df.dropna(how='all')
df = df.dropna(
    subset=COLUMNAS_VACIAS,
    how='all'
)

In [6]:
df[COL_NOMBRE] = (
    df[COL_NOMBRE]
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.translate(TABLA_TILDES)
    .str.replace(r'[^A-ZÑ ]', '', regex=True)
)


In [7]:
# Convierte a numérico, lo inválido → NaN
df[COL_CEDULA] = (df[COL_CEDULA]
    .astype('string')
    .str.replace(r'\D', '', regex=True)
    .replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})
)

In [10]:
# Definir la regex para detectar meses en contratos
RE_MES_EN_CONTRATO = r'(?:ENERO|FEBRERO|MARZO|ABRIL|MAYO|JUNIO|JULIO|AGOSTO|SEPTIEMBRE|OCTUBRE|NOVIEMBRE|DICIEMBRE)'


df[COL_CONTRATO] = (df[COL_CONTRATO]
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    # CAMBIO:"2.019" → "2019"
    .str.replace(r'\b((?:19|20)\d)\.\s*(\d{3})\b', r'\1\2', regex=True)

    # === FILTRAR FECHAS Y TEXTOS NO VÁLIDOS ===
    .mask(
        lambda x: (
            x.str.contains(RE_MES_EN_CONTRATO, na=False) |
            x.str.fullmatch(r'SIN\s+N[UÚ]MERO', na=False, flags=re.IGNORECASE)
        ),
        pd.NA
    )

    # === CASOS ESPECIALES (van PRIMERO prioridad) ===

    # 1. Año mal escrito de 5 dígitos: "1980-20204" → "1980"
    .str.replace(r'^(\d+)-20\d{3}$', r'\1', regex=True)

    # 2. Contratos con sufijo OS: "1723 - OS2" → "1723"
    .str.replace(r'^(\d+)\s*[-–]\s*OS\d+$', r'\1', regex=True, flags=re.IGNORECASE)

    # 3. Contratos LPA: "LPA NO. 001 DE 2023..." → "LPA NO. 001"
    .str.replace(r'^(LPA\s+NO\.?\s*\d+).*$', r'\1', regex=True, flags=re.IGNORECASE)

    # CAMBIO: casos 4 y 6 combinados → terciarios con año (con o sin "No.")
    # "No. 3-1-85910-002 de 2019" → "3185910002"
    # "3-1-86011-001 de 2019"→ "3186011001"
    .str.replace(r'(?:No\.?\s*)?3[-–]1[-–](\d+)[-–](\d+)\s+de\s+\d{4}', r'31\1\2', regex=True, flags=re.IGNORECASE)

    # 5. Terciarios con "No." sin año: "No. 3177258011" → "3177258011"
    .str.replace(r'No\.?\s*(\d{10,})', r'\1', regex=True)

    # 7. Terciarios sin año: "3-1-108847-001" → "31108847001"
    .str.replace(r'3[-–]1[-–](\d+)[-–](\d+)', r'31\1\2', regex=True)

    # 8. Contrato Fiducia con número concatenado: "Contrato Fiducia mercantil 31108847001" → "31108847001"
    .str.replace(r'.*?(\d{10,}).*', r'\1', regex=True)

    # CAMBIO: casos 9 y 10 combinados → sufijo ANT con o sin segundo número
    # "013597-0628881 ANT" → "013597"
    # "05202 ANT"→ "05202"
    .str.replace(r'^(\d+)(?:[-–]\d+)?\s+ANT$', r'\1', regex=True, flags=re.IGNORECASE)

    # === CASOS GENERALES ===

    # 11. Número con año completo (guion, slash): "984-2025", "984/2025" → "984"
    .str.replace(r'^(\d+)\s*[-–/]\s*(?:19|20)\d{2}$', r'\1', regex=True)

    # 12. Número con "DE" y año: "984 DE 2025" → "984"
    .str.replace(r'^(\d+)\s+DE\s+(?:19|20)\d{2}$', r'\1', regex=True, flags=re.IGNORECASE)

    # 13. Número con año corto: "984-25" → "984"
    .str.replace(r'^(\d+)\s*[-–]\s*\d{2}$', r'\1', regex=True)

    # 14. Número seguido de texto: "984 OXI SALENTO..." → "984"
    .str.replace(r'^(\d+)\s+[A-ZÁÉÍÓÚÑ].*', r'\1', regex=True)

    # 15. Eliminar año al final: "1022-2018" → "1022"
    .str.replace(r'-(?:19|20)\d{2}$', '', regex=True)
)

In [11]:
df.to_excel(ARCHIVO_SALIDA, index=False)